In [ ]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [5]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = ['epam/med3web', 'indralab/indra_db', 'choosehappy/HistoQC', 'qulacs/qulacs', 'ropensci/stplanr', 'ZhaoJ9014/face.evoLVe',
            'rasbt/mlxtend', 'systerel/S2OPC', 'teusH/MySense', 'kuwisdelu/Cardinal']
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,00225b0980e17ccd8abb39571feecbbb98f9e60f,epam_med3web


In [6]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

  4%|███▎                                                                           | 137/3230 [02:19<52:30,  1.02s/it]

Got Errors {'027e8d769e0bcc38842ee526d2fa3fa55d0dbdc6': 'Key 027e8d769e0bcc38842ee526d2fa3fa55d0dbdc6 not found in /da5_fast/All.sha1c/commit_2.tch'}


  8%|██████▎                                                                        | 257/3230 [04:21<50:27,  1.02s/it]

Got Errors {'526338b1a28eeeca5bfed5695735e4d222d4077d': 'Key 526338b1a28eeeca5bfed5695735e4d222d4077d not found in /da5_fast/All.sha1c/commit_82.tch'}


 13%|██████████▏                                                                    | 418/3230 [07:05<47:45,  1.02s/it]

...

Got Errors {'fffe0301e3a9b20ffe2ef00f1dc349366a37409a': 'Key fffe0301e3a9b20ffe2ef00f1dc349366a37409a not found in /da5_fast/All.sha1c/commit_127.tch'}


 96%|███████████████████████████████████████████████████████████████████████████▏  | 3114/3230 [52:52<01:58,  1.02s/it]

Got Errors {'e6d557ff705c2fdcb815921fee93c7ec4e83fba1': 'Key e6d557ff705c2fdcb815921fee93c7ec4e83fba1 not found in /da5_fast/All.sha1c/commit_102.tch'}


100%|██████████████████████████████████████████████████████████████████████████████| 3230/3230 [54:50<00:00,  1.02s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,00225b0980e17ccd8abb39571feecbbb98f9e60f,88f5ec8f327d5ca7a6ab1daf6901be26ae7574bd,[bb72c3617d90c63082596d5e65356398f26a3e81],Vladislav Shubnikov <Vladislav_Shubnikov@epam....,1526638320,+0300,Vladislav Shubnikov <Vladislav_Shubnikov@epam....,1526638320,+0300,add ActiveVolume functionality (skull detect a...,00225b0980e17ccd8abb39571feecbbb98f9e60f,epam_med3web
1,0042d7f2465621277a4d11e5995304390e1c82b6,a21c656d6f845face56943d969638723a8ef9281,"[313f0a1e535973004941a8ae0664fb19e95d0cc3, 6c2...",DanilRostov <31970322+DanilRostov@users.norepl...,1684534336,-0700,GitHub <noreply@github.com>,1684534336,-0700,Merge pull request #153 from epam/feature/impr...,0042d7f2465621277a4d11e5995304390e1c82b6,epam_med3web


In [7]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '00225b0980e17ccd8abb39571feecbbb98f9e60f', 'tree': '88f5ec8f327d5ca7a6ab1daf6901be26ae7574bd', 'parent': ['bb72c3617d90c63082596d5e65356398f26a3e81'], 'author': 'Vladislav Shubnikov <Vladislav_Shubnikov@epam.com>', 'author_time': 1526638320, 'author_tz': '+0300', 'committer': 'Vladislav Shubnikov <Vladislav_Shubnikov@epam.com>', 'committer_time': 1526638320, 'committer_tz': '+0300', 'message': 'add ActiveVolume functionality (skull detect and remove) in early development stage (will be migrated from c source)\n\n- Add basic geometry class for active surface iterations\n- Fix some coding standard issues\n'}


In [8]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  project = df.loc[df['sha1'] == k['commit'], 'project'].iloc[0]
  row = pd.Series({'project':project, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [12]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,epam_med3web,00225b0980e17ccd8abb39571feecbbb98f9e60f,Vladislav Shubnikov <Vladislav_Shubnikov@epam....,1526638320,add ActiveVolume functionality (skull detect a...


In [16]:
#mode a means append, so you have all your projects in the same file
yournetid='pvickery'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

In [5]:

project_summary = df_commit_data.groupby('project').agg(
    commits=('commit', 'count'),
    authors=('author', 'nunique'),
    max_time=('author_time', 'max'),
    min_time=('author_time', 'min')
).reset_index()
project_summary['max_time'] = pd.to_datetime(
    project_summary['max_time'], unit='s', utc=True
)

project_summary['min_time'] = pd.to_datetime(
    project_summary['min_time'], unit='s', utc=True
)
project_summary

,project,commits,authors,max_time,min_time
0,choosehappy_histoqc,610,43,2026-04-13 20:00:04+00:00,2017-10-16 13:29:10+00:00
1,epam_med3web,1327,79,2025-08-21 01:14:36+00:00,2017-12-25 15:02:37+00:00
2,indralab_indra_db,3888,24,2025-10-16 21:51:36+00:00,2018-09-20 21:16:31+00:00
3,kuwisdelu_cardinal,1036,29,2025-10-14 18:19:22+00:00,2013-11-17 11:46:32+00:00
4,qulacs_qulacs,3803,113,2025-09-01 00:44:13+00:00,2018-10-05 05:24:32+00:00
5,rasbt_mlxtend,3288,213,2026-02-27 08:53:23+00:00,2014-08-14 01:56:16+00:00
6,ropensci_stplanr,2753,50,2025-05-03 08:48:58+00:00,2015-01-30 08:36:50+00:00
7,systerel_s2opc,13436,51,2025-11-03 13:31:44+00:00,2016-03-14 15:11:00+00:00
8,teush_mysense,1364,6,2025-04-09 15:25:56+00:00,2017-02-22 14:51:50+00:00
9,zhaoj9014_face.evolve,750,11,2025-03-20 02:45:06+00:00,2019-01-04 02:22:17+00:00


# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

| Project | # Commits (WoC) | # Authors (WoC) | Max Time (WoC) | Min Time (WoC) | GitHub Stars | GitHub Forks | GitHub Last Commit |
|---|---:|---:|---|---|---:|---:|---|
| choosehappy_histoqc | 610 | 43 | 2026-04-13 20:00:04+00:00 | 2017-10-16 13:29:10+00:00 | 331 | 118 | September 16, 2026 |
| epam_med3web | 1327 | 79 | 2025-08-21 01:14:36+00:00 | 2017-12-25 15:02:37+00:00 | 368 | 105 | September 24, 2026 |
| indralab_indra_db | 3888 | 24 | 2025-10-16 21:51:36+00:00 | 2018-09-20 21:16:31+00:00 | 22 | 12 | July 9, 2026 |
| kuwisdelu_cardinal | 1036 | 29 | 2025-10-14 18:19:22+00:00 | 2013-11-17 11:46:32+00:00 | 81 | 18 | July 30, 2026 |
| qulacs_qulacs | 3803 | 113 | 2025-09-01 00:44:13+00:00 | 2018-10-05 05:24:32+00:00 | 529 | 147 | September 29, 2026 |
| rasbt_mlxtend | 3288 | 213 | 2026-02-27 08:53:23+00:00 | 2014-08-14 01:56:16+00:00 | 5200 | 920 | September 29, 2026 |
| ropensci_stplanr | 2753 | 50 | 2025-05-03 08:48:58+00:00 | 2015-01-30 08:36:50+00:00 | 443 | 68 | April 17, 2025 |
| systerel_s2opc | 13436 | 51 | 2025-11-03 13:31:44+00:00 | 2016-03-14 15:11:00+00:00 | 28 | 6 | September 30, 2026 |
| teush_mysense | 1364 | 6 | 2025-04-09 15:25:56+00:00 | 2017-02-22 14:51:50+00:00 | 71 | 18 | September 1, 2026 |
| zhaoj9014_face.evolve | 750 | 11 | 2025-03-20 02:45:06+00:00 | 2019-01-04 02:22:17+00:00 | 3600 | 759 | March 19, 2025 |
